In [ ]:
import requests
import json
from datetime import datetime

# 공공데이터포털에서 발급받은 일반 인증키 (Encoding)
# 실제 사용 시에는 os.getenv 등을 사용하여 안전하게 관리하는 것이 좋습니다.
service_key = "YOUR_SERVICE_KEY"  # 여기에 발급받은 인증키를 붙여넣으세요.

# 데이터를 요청할 URL
url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"

# 오늘 날짜와 예보 발표 시간을 설정
# 기상청 API는 특정 시간에 예보를 발표하므로, 가장 가까운 발표 시간을 찾아야 합니다.
# 예: 02:00, 05:00, 08:00, 11:00, 14:00, 17:00, 20:00, 23:00
now = datetime.now()
base_date = now.strftime("%Y%m%d")
# 현재 시간에 따라 가장 가까운 base_time을 설정해야 하지만, 간단한 예시를 위해 '0800'으로 고정합니다.
base_time = "0800"

# 상일동의 기상청 격자 좌표 (X, Y)
# 참고: https://www.data.go.kr/data/15084084/fileData.do 에서 '격자x,y초단기실황조회' 파일 참고
nx = 63
ny = 127

# API 요청에 필요한 파라미터 설정
params = {
    "serviceKey": service_key,
    "pageNo": "1",
    "numOfRows": "100",  # 하루의 모든 정보를 가져오기 위해 넉넉하게 설정
    "dataType": "JSON",
    "base_date": base_date,
    "base_time": base_time,
    "nx": str(nx),
    "ny": str(ny),
}

try:
    # API 요청 보내기
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()  # 오류가 발생하면 예외를 발생시킴

    # 응답 데이터 파싱
    data = response.json()
    items = data["response"]["body"]["items"]["item"]

    # 필요한 정보(예: 현재 시간의 기온, 하늘 상태)만 추출하여 출력
    print(f"--- {base_date} {base_time} 기준 상일동 날씨 예보 ---")

    weather_info = {}
    for item in items:
        # fcstTime은 예보 시간, category는 정보의 종류
        fcst_time = item["fcstTime"]
        if fcst_time not in weather_info:
            weather_info[fcst_time] = {}

        category = item["category"]
        value = item["fcstValue"]

        # 카테고리 코드에 따라 정보 저장
        if category == "TMP":  # 1시간 기온
            weather_info[fcst_time]["기온"] = f"{value}°C"
        elif category == "SKY":  # 하늘 상태 (1:맑음, 3:구름많음, 4:흐림)
            sky_dict = {"1": "맑음 ☀️", "3": "구름많음 🌥️", "4": "흐림 ☁️"}
            weather_info[fcst_time]["하늘"] = sky_dict.get(value, "정보없음")
        elif category == "PTY":  # 강수 형태 (0:없음, 1:비, 2:비/눈, 3:눈, 4:소나기)
            pty_dict = {
                "0": "없음",
                "1": "비 🌧️",
                "2": "비/눈",
                "3": "눈 ❄️",
                "4": "소나기 🌦️",
            }
            weather_info[fcst_time]["강수"] = pty_dict.get(value, "정보없음")
        elif category == "POP":  # 강수 확률
            weather_info[fcst_time]["강수확률"] = f"{value}%"

    # 예보 시간 순으로 정렬하여 출력
    for time, info in sorted(weather_info.items()):
        print(
            f"[{time[:2]}시] {info.get('기온', '')}, 하늘: {info.get('하늘', '')}, 강수확률: {info.get('강수확률', '')}"
        )


except requests.exceptions.RequestException as e:
    print(f"API 요청 중 오류가 발생했습니다: {e}")
except (KeyError, TypeError) as e:
    print(f"데이터 파싱 중 오류가 발생했습니다. 응답 형식을 확인해주세요: {e}")
    # print("받은 데이터:", response.text) # 디버깅 시 확인용